# 02 — Exploratory Data Analysis (EDA)

This notebook explores the raw Idealista dataset collected in `01_data_collection.ipynb`.

Main goals:
- Load the latest raw dataset from `data/raw/`
- Understand dataset structure and available features
- Check missing values, duplicates, and suspicious values
- Explore price, surface, rooms, bathrooms, and location patterns
- Create first business insights for the investment opportunity project
- Prepare hypotheses for cleaning, feature engineering, and model training


## 1. Imports and project paths


In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data directory:", RAW_DATA_DIR.resolve())
print("Processed data directory:", PROCESSED_DATA_DIR.resolve())


Project root: C:\Users\hugol\Desktop\Python\B2\ml-poc-project
Raw data directory: C:\Users\hugol\Desktop\Python\B2\ml-poc-project\data\raw
Processed data directory: C:\Users\hugol\Desktop\Python\B2\ml-poc-project\data\processed


## 2. Load latest raw dataset

By default, this notebook loads the latest file matching:

```text
data/raw/idealista_raw_listings_*.csv
```

If needed, manually replace `csv_files[-1]` with a specific file path.


In [2]:
csv_files = sorted(RAW_DATA_DIR.glob("idealista_raw_listings_*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No raw CSV file found in {RAW_DATA_DIR}. "
        "Run notebooks/01_data_collection.ipynb first."
    )

RAW_CSV_PATH = csv_files[-1]
print("Loading:", RAW_CSV_PATH)

df = pd.read_csv(RAW_CSV_PATH)
print("Dataset shape:", df.shape)

display(df.head())


Loading: ..\data\raw\idealista_raw_listings_20260430_185730.csv
Dataset shape: (500, 51)


,propertyCode,thumbnail,externalReference,numPhotos,price,propertyType,operation,size,rooms,bathrooms,address,province,municipality,district,country,neighborhood,latitude,longitude,showAddress,url,distance,description,hasVideo,status,newDevelopment,priceByArea,hasPlan,has3DTour,has360,hasStaging,notes,topNewDevelopment,newDevelopmentHighlight,topPlus,priceInfo.price.amount,priceInfo.price.currencySuffix,parkingSpace.hasParkingSpace,parkingSpace.isParkingSpaceIncludedInPrice,detailedType.typology,detailedType.subTypology,suggestedTexts.subtitle,suggestedTexts.title,highlight.groupDescription,floor,exterior,hasLift,priceInfo.price.priceDropInfo.formerPrice,priceInfo.price.priceDropInfo.priceDropValue,priceInfo.price.priceDropInfo.priceDropPercentage,parkingSpace.parkingSpacePrice,newDevelopmentFinished
0,111308260,https://img4.idealista.com/blur/480_360_mq/0/i...,CA219287,62,"3,950,000.00",chalet,sale,603.00,7,7,calle Guarda,Madrid,Pozuelo de Alarcón,Somosaguas,es,Somosaguas,40.43,-3.78,False,https://www.idealista.com/inmueble/111308260/,6587,"En la exclusiva urbanización El Montecillo, en...",True,good,False,"6,551.00",True,True,False,False,[],False,False,True,"3,950,000.00",€,True,True,chalet,independantHouse,"Somosaguas, Pozuelo de Alarcón",Casa independiente en calle Guarda,Top+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,110715434,https://img4.idealista.com/blur/480_360_mq/0/i...,EC216671,46,"1,400,000.00",penthouse,sale,177.00,2,2,calle de Isabel la Católica,Madrid,Madrid,Centro,es,Palacio,40.42,-3.71,False,https://www.idealista.com/inmueble/110715434/,617,GILMAR Consulting Inmobiliario pone a su dispo...,True,good,False,"7,910.00",True,True,False,True,[],False,False,True,"1,400,000.00",€,NaN,NaN,flat,penthouse,"Palacio, Madrid",Ático en calle de Isabel la Católica,Top+,6,True,True,"1,500,000.00","100,000.00",7.00,NaN,NaN
2,110921969,https://img4.idealista.com/blur/480_360_mq/0/i...,NY-219592,71,"3,800,000.00",chalet,sale,"1,000.00",7,6,Barrio Somosaguas,Madrid,Pozuelo de Alarcón,Somosaguas,es,Somosaguas,40.43,-3.79,False,https://www.idealista.com/inmueble/110921969/,7775,SIN HONORARIOS A LA PARTE COMPRADORA GILMAR se...,True,renew,False,"3,800.00",True,True,False,True,[],False,False,True,"3,800,000.00",€,True,True,chalet,independantHouse,"Somosaguas, Pozuelo de Alarcón",Casa independiente,Top+,NaN,NaN,NaN,"4,200,000.00","400,000.00",10.00,NaN,NaN
3,109999723,https://img4.idealista.com/blur/480_360_mq/0/i...,DS-P22798,33,"2,100,000.00",flat,sale,137.00,3,3,calle Alcalá,Madrid,Madrid,Barrio de Salamanca,es,Goya,40.42,-3.68,False,https://www.idealista.com/inmueble/109999723/,2404,DIZA Consultores les presenta este fantástico ...,True,good,False,"15,328.00",True,True,False,True,[],False,False,True,"2,100,000.00",€,NaN,NaN,flat,NaN,"Goya, Madrid",Piso en calle Alcalá,Top+,3,True,True,"2,300,000.00","200,000.00",9.00,NaN,NaN
4,103883825,https://img4.idealista.com/blur/480_360_mq/0/i...,AS192516,51,"4,950,000.00",flat,sale,383.00,5,3,calle Maldonado,Madrid,Madrid,Barrio de Salamanca,es,Castellana,40.43,-3.68,False,https://www.idealista.com/inmueble/103883825/,2523,En el corazón del prestigioso barrio de Salama...,True,good,False,"12,924.00",True,True,False,True,[],False,False,True,"4,950,000.00",€,NaN,NaN,flat,NaN,"Castellana, Madrid",Piso en calle Maldonado,Top+,6,True,True,NaN,NaN,NaN,NaN,NaN


## 3. Dataset overview


In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique_values": df.nunique(dropna=True).values
}).sort_values("missing_pct", ascending=False)

display(overview)


Rows: 500
Columns: 51


,column,dtype,missing_count,missing_pct,unique_values
50,newDevelopmentFinished,object,495,99.00,2
49,parkingSpace.parkingSpacePrice,float64,483,96.60,9
46,priceInfo.price.priceDropInfo.formerPrice,float64,411,82.20,68
48,priceInfo.price.priceDropInfo.priceDropPercentage,float64,411,82.20,12
47,priceInfo.price.priceDropInfo.priceDropValue,float64,411,82.20,37
39,detailedType.subTypology,object,340,68.00,6
36,parkingSpace.hasParkingSpace,object,259,51.80,1
37,parkingSpace.isParkingSpaceIncludedInPrice,object,259,51.80,2
43,floor,object,100,20.00,16
44,exterior,object,95,19.00,2


## 4. Key columns check

The project mainly needs variables related to:
- price
- size
- location
- property characteristics
- amenities
- listing identifiers


In [ ]:
expected_columns = [
    "propertyCode", "price", "size", "rooms", "bathrooms",
    "district", "neighborhood", "municipality", "province",
    "propertyType", "operation", "url",
    "latitude", "longitude",
    "floor", "hasLift", "exterior", "parkingSpace.hasParkingSpace"
]

available_expected = [col for col in expected_columns if col in df.columns]
missing_expected = [col for col in expected_columns if col not in df.columns]

print("Available expected columns:")
print(available_expected)

print("\nMissing expected columns:")
print(missing_expected)

display(df[available_expected].head())


## 5. Duplicate listings

Listings can appear more than once across API pages or collection runs.  
The main unique identifier is usually `propertyCode`.


In [ ]:
if "propertyCode" in df.columns:
    duplicate_count = df["propertyCode"].duplicated().sum()
    duplicate_pct = duplicate_count / len(df) * 100
    print(f"Duplicate propertyCode count: {duplicate_count}")
    print(f"Duplicate percentage: {duplicate_pct:.2f}%")

    if duplicate_count > 0:
        display(df[df["propertyCode"].duplicated(keep=False)].sort_values("propertyCode").head(20))
else:
    print("propertyCode column not found.")


## 6. Basic type cleaning for EDA

This is not the final cleaning pipeline.  
The goal is only to make exploratory analysis easier.


In [ ]:
df_eda = df.copy()

numeric_candidates = [
    "price", "size", "rooms", "bathrooms", "latitude", "longitude"
]

for col in numeric_candidates:
    if col in df_eda.columns:
        df_eda[col] = pd.to_numeric(df_eda[col], errors="coerce")

if "propertyCode" in df_eda.columns:
    before = len(df_eda)
    df_eda = df_eda.drop_duplicates(subset=["propertyCode"], keep="first")
    after = len(df_eda)
    print(f"Dropped {before - after} duplicated listings for EDA.")

print("EDA dataset shape:", df_eda.shape)


## 7. Basic statistics


In [ ]:
numeric_cols = [col for col in ["price", "size", "rooms", "bathrooms"] if col in df_eda.columns]
display(df_eda[numeric_cols].describe())

categorical_cols = [col for col in ["district", "neighborhood", "municipality", "province", "propertyType"] if col in df_eda.columns]

for col in categorical_cols:
    print(f"\nTop values for {col}:")
    display(df_eda[col].value_counts(dropna=False).head(15))


## 8. Create first derived feature: price per square meter

`price_per_m2` is one of the most important real estate indicators.


In [ ]:
if "price" in df_eda.columns and "size" in df_eda.columns:
    df_eda["price_per_m2"] = np.where(
        df_eda["size"] > 0,
        df_eda["price"] / df_eda["size"],
        np.nan
    )

    display(df_eda[["price", "size", "price_per_m2"]].describe())
else:
    print("price and/or size columns not found.")


## 9. Outlier inspection

We do **not** remove outliers permanently here.  
We only create an EDA-filtered dataset to make charts easier to interpret.


In [ ]:
df_plot = df_eda.copy()

if "price" in df_plot.columns:
    df_plot = df_plot[df_plot["price"].between(50_000, 5_000_000)]

if "size" in df_plot.columns:
    df_plot = df_plot[df_plot["size"].between(20, 500)]

if "price_per_m2" in df_plot.columns:
    df_plot = df_plot[df_plot["price_per_m2"].between(500, 25_000)]

print("Original EDA shape:", df_eda.shape)
print("Plot-filtered shape:", df_plot.shape)


## 10. Raw data visualization — price distribution

This visualization helps identify price dispersion and extreme values.


In [ ]:
if "price" in df_plot.columns:
    plt.figure(figsize=(10, 6))
    plt.hist(df_plot["price"].dropna(), bins=40)
    plt.title("Distribution of listing prices")
    plt.xlabel("Price (€)")
    plt.ylabel("Number of listings")
    plt.ticklabel_format(style="plain", axis="x")
    plt.tight_layout()

    fig_path = REPORTS_DIR / "eda_price_distribution.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()

    print("Saved figure:", fig_path)


## 11. Surface distribution


In [ ]:
if "size" in df_plot.columns:
    plt.figure(figsize=(10, 6))
    plt.hist(df_plot["size"].dropna(), bins=40)
    plt.title("Distribution of property size")
    plt.xlabel("Size (m²)")
    plt.ylabel("Number of listings")
    plt.tight_layout()

    fig_path = REPORTS_DIR / "eda_size_distribution.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()

    print("Saved figure:", fig_path)


## 12. Feature-engineered visualization — price per m² by district

This is central to the business problem: location strongly affects fair market value.


In [ ]:
if "district" in df_plot.columns and "price_per_m2" in df_plot.columns:
    count_col = "propertyCode" if "propertyCode" in df_plot.columns else "price_per_m2"
    district_summary = (
        df_plot
        .groupby("district")
        .agg(
            listings=(count_col, "count"),
            avg_price=("price", "mean"),
            median_price=("price", "median"),
            avg_size=("size", "mean"),
            avg_price_per_m2=("price_per_m2", "mean"),
            median_price_per_m2=("price_per_m2", "median")
        )
        .sort_values("median_price_per_m2", ascending=False)
    )

    display(district_summary.head(20))

    top_districts = district_summary[district_summary["listings"] >= 5].head(15)

    plt.figure(figsize=(12, 7))
    plt.bar(top_districts.index.astype(str), top_districts["median_price_per_m2"])
    plt.title("Median price per m² by district")
    plt.xlabel("District")
    plt.ylabel("Median price per m² (€)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    fig_path = REPORTS_DIR / "eda_price_per_m2_by_district.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()

    print("Saved figure:", fig_path)
else:
    print("district or price_per_m2 column not found.")


## 13. Relationship between size and price


In [ ]:
if "size" in df_plot.columns and "price" in df_plot.columns:
    plt.figure(figsize=(10, 6))
    plt.scatter(df_plot["size"], df_plot["price"], alpha=0.5)
    plt.title("Relationship between size and price")
    plt.xlabel("Size (m²)")
    plt.ylabel("Price (€)")
    plt.ticklabel_format(style="plain", axis="y")
    plt.tight_layout()

    fig_path = REPORTS_DIR / "eda_size_vs_price.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()

    print("Saved figure:", fig_path)


## 14. Rooms and bathrooms analysis


In [ ]:
for col in ["rooms", "bathrooms"]:
    if col in df_plot.columns and "price" in df_plot.columns:
        agg_dict = {
            "listings": ("price", "count"),
            "median_price": ("price", "median")
        }
        if "price_per_m2" in df_plot.columns:
            agg_dict["median_price_per_m2"] = ("price_per_m2", "median")

        grouped = df_plot.groupby(col).agg(**agg_dict).sort_index()
        print(f"\nPrice analysis by {col}:")
        display(grouped)

        plt.figure(figsize=(8, 5))
        plt.bar(grouped.index.astype(str), grouped["median_price"])
        plt.title(f"Median price by number of {col}")
        plt.xlabel(col)
        plt.ylabel("Median price (€)")
        plt.ticklabel_format(style="plain", axis="y")
        plt.tight_layout()

        fig_path = REPORTS_DIR / f"eda_median_price_by_{col}.png"
        plt.savefig(fig_path, dpi=150)
        plt.show()

        print("Saved figure:", fig_path)


## 15. Amenities analysis

Depending on the API response, amenities may appear as columns such as:
- `hasLift`
- `exterior`
- `parkingSpace.hasParkingSpace`
- `hasVideo`
- `hasPlan`


In [ ]:
amenity_candidates = [
    "hasLift",
    "exterior",
    "parkingSpace.hasParkingSpace",
    "hasVideo",
    "hasPlan",
    "has3DTour",
    "newDevelopment"
]

amenity_cols = [col for col in amenity_candidates if col in df_plot.columns]

print("Available amenity columns:")
print(amenity_cols)

for col in amenity_cols:
    print(f"\n{col} value counts:")
    display(df_plot[col].value_counts(dropna=False))

    if "price_per_m2" in df_plot.columns:
        amenity_summary = (
            df_plot
            .groupby(col)
            .agg(
                listings=("price_per_m2", "count"),
                median_price_per_m2=("price_per_m2", "median"),
                median_price=("price", "median")
            )
        )
        display(amenity_summary)


## 16. Correlation analysis

This helps identify numerical variables that may be useful for prediction.


In [ ]:
corr_candidates = [
    "price", "size", "rooms", "bathrooms", "price_per_m2",
    "latitude", "longitude"
]
corr_cols = [col for col in corr_candidates if col in df_plot.columns]

if len(corr_cols) >= 2:
    corr = df_plot[corr_cols].corr(numeric_only=True)
    display(corr)

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation matrix")
    plt.tight_layout()

    fig_path = REPORTS_DIR / "eda_correlation_matrix.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()

    print("Saved figure:", fig_path)
else:
    print("Not enough numerical columns for correlation analysis.")


## 17. Optional map visualization

If latitude and longitude are available, this map gives a first view of spatial patterns.

If Plotly is not installed:

```bash
pip install plotly
```


In [ ]:
if {"latitude", "longitude"}.issubset(df_plot.columns):
    try:
        import plotly.express as px

        map_df = df_plot.dropna(subset=["latitude", "longitude", "price"])

        hover_cols = [col for col in ["price", "size", "rooms", "bathrooms", "price_per_m2", "url"] if col in map_df.columns]

        fig = px.scatter_mapbox(
            map_df,
            lat="latitude",
            lon="longitude",
            color="price_per_m2" if "price_per_m2" in map_df.columns else "price",
            size="price",
            hover_name="district" if "district" in map_df.columns else None,
            hover_data=hover_cols,
            zoom=10,
            height=600,
            title="Idealista listings map"
        )
        fig.update_layout(mapbox_style="open-street-map")
        fig.update_layout(margin={"r":0, "t":50, "l":0, "b":0})
        fig.show()
    except ImportError:
        print("Plotly is not installed. Run: pip install plotly")
else:
    print("latitude and/or longitude not available.")


## 18. Description keyword analysis

This simple text analysis identifies business-relevant keywords in listing descriptions.


In [ ]:
description_col = None
for candidate in ["description", "propertyComment", "suggestedTexts.subtitle"]:
    if candidate in df_eda.columns:
        description_col = candidate
        break

keywords = [
    "reformado", "terraza", "ático", "luminoso", "exterior",
    "garaje", "ascensor", "lujo", "exclusivo", "oportunidad",
    "rentabilidad", "inversión", "piscina", "balcón"
]

if description_col:
    print("Using description column:", description_col)

    text_series = df_eda[description_col].fillna("").astype(str).str.lower()

    keyword_counts = {}
    for kw in keywords:
        keyword_counts[kw] = text_series.str.contains(kw, regex=False).sum()

    keyword_df = (
        pd.DataFrame.from_dict(keyword_counts, orient="index", columns=["count"])
        .sort_values("count", ascending=False)
    )

    display(keyword_df)

    plt.figure(figsize=(10, 6))
    plt.bar(keyword_df.index, keyword_df["count"])
    plt.title("Keyword frequency in listing descriptions")
    plt.xlabel("Keyword")
    plt.ylabel("Number of listings")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    fig_path = REPORTS_DIR / "eda_description_keywords.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()

    print("Saved figure:", fig_path)
else:
    print("No description-like column found.")


## 19. First investment-opportunity proxy

This is not the final model.  
It is a first business-oriented indicator for exploration:

- Compute average price per m² by district
- Compare each listing against its district average
- Negative values may indicate potential underpricing


In [ ]:
if "district" in df_eda.columns and "price_per_m2" in df_eda.columns:
    df_eda["district_avg_price_per_m2"] = (
        df_eda.groupby("district")["price_per_m2"].transform("mean")
    )

    df_eda["discount_vs_district_avg_pct"] = (
        (df_eda["price_per_m2"] - df_eda["district_avg_price_per_m2"])
        / df_eda["district_avg_price_per_m2"]
        * 100
    )

    opportunity_cols = [
        "propertyCode", "district", "price", "size", "rooms", "bathrooms",
        "price_per_m2", "district_avg_price_per_m2",
        "discount_vs_district_avg_pct", "url"
    ]
    opportunity_cols = [col for col in opportunity_cols if col in df_eda.columns]

    opportunities = (
        df_eda
        .dropna(subset=["discount_vs_district_avg_pct"])
        .sort_values("discount_vs_district_avg_pct")
        [opportunity_cols]
        .head(20)
    )

    display(opportunities)
else:
    print("district or price_per_m2 column not available.")


## 20. Export EDA-enriched dataset

This dataset is not the final cleaned modeling dataset.  
It includes useful EDA features that will guide the next notebook.


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

eda_output_path = PROCESSED_DATA_DIR / f"idealista_eda_enriched_{timestamp}.csv"
df_eda.to_csv(eda_output_path, index=False, encoding="utf-8-sig")

print("EDA-enriched dataset saved to:", eda_output_path)
print("Shape:", df_eda.shape)


## 21. Initial findings to write in `assignment1.md`

Use this section to summarize your first observations after running the notebook.

Suggested structure:

```markdown
## First EDA Findings

- The raw dataset contains X listings and Y columns.
- The most important variables available are price, size, rooms, bathrooms, district and coordinates.
- Price distribution is highly skewed, with several expensive outliers.
- Price per square meter varies significantly by district, confirming that location is a key driver of property value.
- Some listings have missing values for amenities or location-related fields.
- Duplicate listings were checked using `propertyCode`.
- The first investment hypothesis is that properties priced significantly below their district average price per m² may represent potential investment opportunities.
```
